# Deploying a Machine Learning Model: A Beginner-Friendly Guide

**Three practical ways to put a model in front of users**
1. **Streamlit** — turn Python code into an interactive web app quickly.
2. **FastAPI** — expose your model through an API that other apps can call.
3. **Gradio** — build a simple interactive demo with very little UI code.

This notebook explains not only *what* to do, but **why** each step matters. It uses a small scikit-learn classification model that predicts a flower species from four measurements, so the whole workflow can run on an ordinary computer.

> **Important:** This is a learning example, not a production system. Do not deploy private data or an unreviewed model publicly.

## Learning goals

By the end, you should be able to explain:

- what model deployment means;
- why a saved model file needs preprocessing and matching libraries;
- how to save and load a model safely;
- how to create a Streamlit app, a FastAPI endpoint, and a Gradio demo;
- how to test the app locally and prepare it for hosting;
- common errors and how to troubleshoot them.

### The big picture

Training creates a model. Deployment makes that trained model usable by someone or something else.

`data → train model → evaluate → save model → load model in app/API → user sends input → prediction is returned`

## 1. What does “deploying a model” mean?

Imagine you trained a model in a notebook. You can call `model.predict(...)` there, but a user cannot automatically access your notebook.

Deployment wraps the model in a usable interface:
- a **web page** with boxes and buttons;
- an **API** that receives structured data and returns structured results; or
- an **interactive demo** for quick testing.

### Why are we learning three options?

| Tool | Best for | Example |
|---|---|---|
| Streamlit | Data apps and simple dashboards | A user enters measurements and clicks Predict |
| FastAPI | A backend service for other software | A mobile app sends JSON and receives a prediction |
| Gradio | Quick ML demos | A simple form for testing a model |

They are not exact substitutes. Streamlit and Gradio focus on the user interface; FastAPI focuses on a programmatic API. A real system can use more than one.

## 2. Set up the project

Create a folder named `ml-deployment-beginner`. Inside it, we will eventually have:

```text
ml-deployment-beginner/
├── train_model.py       # trains and saves the model
├── model.joblib         # saved model artifact (created below)
├── app_streamlit.py     # Streamlit web app
├── app_fastapi.py       # FastAPI service
├── app_gradio.py        # Gradio demo
└── requirements.txt     # packages needed to run the project
```

**Why separate files?** Each file has one main job. Training code creates the model; app files load and use it. This is easier to maintain than putting everything into one huge file.

### Install the packages

Run this in a terminal, not in a Python code cell:

```bash
python -m venv .venv
```

Activate the environment:

- **Windows:** `.venv\Scripts\activate`
- **macOS/Linux:** `source .venv/bin/activate`

Then install:

```bash
python -m pip install --upgrade pip
python -m pip install scikit-learn joblib pandas streamlit fastapi "uvicorn[standard]" gradio
```

**Why use a virtual environment?** It keeps this project's packages separate from other Python projects and reduces version conflicts.

If you are using an online notebook service, you may need to run installation commands in a cell with `!` at the beginning. Local terminal commands and notebook commands are not always interchangeable.

## 3. Train a small example model

You said you want to learn deployment, so we need a model file to deploy. This example uses scikit-learn's built-in Iris dataset. It contains measurements of flowers and their species labels.

**Why use a built-in dataset?** It avoids downloading a file and lets us focus on deployment. Later, you can replace it with your own trained model.

The model will be a **Pipeline** containing a scaler and a classifier. A pipeline saves preprocessing and prediction together. That matters because the same preprocessing must be used during training and prediction.

In [1]:
# !pip install gradio
from pathlib import Path
import joblib
import sklearn
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Load example data
iris = load_iris()
X = iris.data
y = iris.target

# Keep some data aside so we can test on examples the model did not train on
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Put preprocessing and the model into one object
model = make_pipeline(
    StandardScaler(),
    RandomForestClassifier(n_estimators=100, random_state=42)
)

# Learn patterns from the training data
model.fit(X_train, y_train)

# Evaluate on the held-out test data
predictions = model.predict(X_test)
print("Accuracy:", round(accuracy_score(y_test, predictions), 3))
print(classification_report(y_test, predictions, target_names=iris.target_names))

Accuracy: 0.9
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       0.82      0.90      0.86        10
   virginica       0.89      0.80      0.84        10

    accuracy                           0.90        30
   macro avg       0.90      0.90      0.90        30
weighted avg       0.90      0.90      0.90        30



### Read the training code in simple English

- `X` contains the four flower measurements; `y` contains the correct species labels.
- `train_test_split` sets aside test examples. **Why?** Testing only on training examples can make a model look better than it really is.
- `StandardScaler` scales the input measurements. We include it in the pipeline so the same transformation is applied later.
- `RandomForestClassifier` is the learning algorithm.
- `fit` trains the pipeline.
- `predict` asks the trained pipeline to predict labels.
- `accuracy_score` reports the fraction of test examples classified correctly. Accuracy is useful, but it is not the only measure of model quality.

**Beginner check:** Which line actually trains the model? Answer: `model.fit(X_train, y_train)`.

## 4. Save the model file

A trained model exists in memory while Python is running. Saving it lets another program load it later without retraining.

We will save:
- the fitted pipeline;
- the class names;
- the feature names, in the correct order;
- the scikit-learn version used during training.

**Why save feature names?** A model trained on `[sepal length, sepal width, petal length, petal width]` expects those measurements in that order. Swapping two inputs can silently produce a wrong prediction.

In [2]:
artifact = {
    "model": model,
    "target_names": iris.target_names.tolist(),
    "feature_names": iris.feature_names,
    "sklearn_version": sklearn.__version__,
}

MODEL_PATH = Path("model.joblib")
joblib.dump(artifact, MODEL_PATH)

print(f"Saved model to: {MODEL_PATH.resolve()}")
print("scikit-learn version used:", sklearn.__version__)

Saved model to: C:\Users\abdulmalik\Downloads\model.joblib
scikit-learn version used: 1.7.2


### Load it back and test it

```python
artifact_loaded = joblib.load("model.joblib")
loaded_model = artifact_loaded["model"]
```

**Security warning:** `joblib` uses Python object serialization. Only load model files from a source you trust. A malicious serialized file can execute harmful code during loading.

Model files can also be version-sensitive. For dependable deployment, use compatible package versions and test the loaded model in the same environment as the app.

In [3]:
loaded_artifact = joblib.load("model.joblib")
loaded_model = loaded_artifact["model"]

# Predict one test flower, using the same four-feature order as training
one_flower = X_test[0].reshape(1, -1)
predicted_index = int(loaded_model.predict(one_flower)[0])

print("Predicted species:", loaded_artifact["target_names"][predicted_index])
print("Actual species:", iris.target_names[y_test[0]])

Predicted species: setosa
Actual species: setosa


## 5. Option A — Build a Streamlit app

Streamlit turns a Python script into an interactive web page. It is beginner-friendly because you can build a UI without writing HTML, CSS, or JavaScript.

### Create `app_streamlit.py`

Copy the following into a file named `app_streamlit.py` in the same folder as `model.joblib`.

In [4]:
%%writefile app_streamlit.py
import joblib
import numpy as np
import streamlit as st
from pathlib import Path

MODEL_PATH = Path("model.joblib")
artifact = joblib.load(MODEL_PATH)
model = artifact["model"]
target_names = artifact["target_names"]

st.set_page_config(page_title="Flower Predictor", page_icon="🌸")
st.title("🌸 Iris Flower Predictor")
st.write(
    "Enter four flower measurements in centimetres. "
    "This is a learning demo, not a scientific identification tool."
)

sepal_length = st.number_input("Sepal length (cm)", min_value=0.0, value=5.1, step=0.1)
sepal_width = st.number_input("Sepal width (cm)", min_value=0.0, value=3.5, step=0.1)
petal_length = st.number_input("Petal length (cm)", min_value=0.0, value=1.4, step=0.1)
petal_width = st.number_input("Petal width (cm)", min_value=0.0, value=0.2, step=0.1)

if st.button("Predict species"):
    features = np.array([[
        sepal_length, sepal_width, petal_length, petal_width
    ]])
    prediction_index = int(model.predict(features)[0])
    st.success(f"Predicted species: {target_names[prediction_index]}")

Writing app_streamlit.py


### Why does this app work?

- `st.number_input` creates a number box.
- `st.button` creates a button and runs the prediction block when clicked.
- `np.array([[...]])` makes a two-dimensional input: scikit-learn expects a batch of rows, even when there is only one example.
- The model file is loaded once when Streamlit runs the script. Streamlit can rerun the script after interactions, so for larger models you may later learn caching with `st.cache_resource`.

### Run the app

In a terminal opened inside the project folder:

```bash
streamlit run app_streamlit.py
```

Streamlit prints a local URL, usually `http://localhost:8501`. Open it in your browser.

**If it says the file cannot be found:** check that `model.joblib` is in the same folder as `app_streamlit.py`, and that you started the command from the correct environment.

### Try it

1. Leave the default values and click **Predict species**.
2. Change one or more measurements.
3. Click Predict again.
4. Stop the app with `Ctrl+C` in the terminal.

## 6. Option B — Serve the model with FastAPI

FastAPI is for building APIs. An API is a defined way for one program to request a service from another program.

For example, a phone app might send:

```json
{
  "sepal_length": 5.1,
  "sepal_width": 3.5,
  "petal_length": 1.4,
  "petal_width": 0.2
}
```

The API responds with JSON containing the predicted species.

**Why use an API?** It separates the model service from the user interface. A website, mobile app, or another backend can all call the same endpoint.

### Create `app_fastapi.py`

Save this code in `app_fastapi.py`, next to `model.joblib`.

In [8]:
%%writefile app_fastapi.py

from pathlib import Path
import joblib
import numpy as np
from fastapi import FastAPI
from pydantic import BaseModel, Field

MODEL_PATH = Path("model.joblib")
artifact = joblib.load(MODEL_PATH)
model = artifact["model"]
target_names = artifact["target_names"]

app = FastAPI(title="Iris Prediction API", version="1.0.0")

class FlowerInput(BaseModel):
    # Field validates that values are numbers greater than zero
    sepal_length: float = Field(gt=0, description="Sepal length in cm")
    sepal_width: float = Field(gt=0, description="Sepal width in cm")
    petal_length: float = Field(gt=0, description="Petal length in cm")
    petal_width: float = Field(gt=0, description="Petal width in cm")

@app.get("/")
def home():
    return {"message": "Iris Prediction API is running"}

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/predict")
def predict(flower: FlowerInput):
    features = np.array([[
        flower.sepal_length,
        flower.sepal_width,
        flower.petal_length,
        flower.petal_width,
    ]])

    prediction_index = int(model.predict(features)[0])
    return {
        "predicted_class_index": prediction_index,
        "predicted_species": target_names[prediction_index],
    }

Overwriting app_fastapi.py


### Understand the important parts

- `FastAPI(...)` creates the API application.
- `BaseModel` defines the expected request format.
- `Field(gt=0)` rejects zero and negative measurements. Validation helps catch bad input early; production systems may need more realistic domain limits too.
- `@app.get("/")` defines a GET endpoint, commonly used to check that the service responds.
- `@app.post("/predict")` defines a POST endpoint for sending prediction input.
- The returned Python dictionary is converted to JSON by FastAPI.

### Run the API

```bash
uvicorn app_fastapi:app --reload
```

Open `http://127.0.0.1:8000/docs`. FastAPI automatically creates an interactive API page where you can try `/predict`.

Example request body:

```json
{
  "sepal_length": 5.1,
  "sepal_width": 3.5,
  "petal_length": 1.4,
  "petal_width": 0.2
}
```

**Why `app_fastapi:app`?** The first `app_fastapi` is the Python filename without `.py`; the second `app` is the variable holding the FastAPI application.

The `--reload` option is useful during development because the server restarts when you change code. For production, use a deployment setup appropriate to your hosting platform instead of relying on development reload mode.

## 7. Option C — Build a Gradio demo

Gradio is another beginner-friendly way to make an interactive machine-learning demo. It is especially handy for sharing a proof of concept or letting people test a model through a simple form.

### Create `app_gradio.py`

In [6]:
%%writefile app_gradio.py
from pathlib import Path
import joblib
import gradio as gr
import numpy as np

MODEL_PATH = Path("model.joblib")
artifact = joblib.load(MODEL_PATH)
model = artifact["model"]
target_names = artifact["target_names"]

def predict_species(sepal_length, sepal_width, petal_length, petal_width):
    features = np.array([[
        sepal_length, sepal_width, petal_length, petal_width
    ]])
    prediction_index = int(model.predict(features)[0])
    return target_names[prediction_index]

demo = gr.Interface(
    fn=predict_species,
    inputs=[
        gr.Number(label="Sepal length (cm)", value=5.1, minimum=0.01),
        gr.Number(label="Sepal width (cm)", value=3.5, minimum=0.01),
        gr.Number(label="Petal length (cm)", value=1.4, minimum=0.01),
        gr.Number(label="Petal width (cm)", value=0.2, minimum=0.01),
    ],
    outputs=gr.Textbox(label="Predicted species"),
    title="Iris Flower Predictor",
    description="A small demo showing how a saved model can be used in an app.",
)

if __name__ == "__main__":
    demo.launch()

Writing app_gradio.py


### Why Gradio?

You provide a Python function (`predict_species`) and describe its inputs and output. Gradio builds the interactive interface around it.

Run it with:

```bash
python app_gradio.py
```

It will print a local address, often `http://127.0.0.1:7860`. Open that address in your browser.

### Compare the three choices

| Question | Streamlit | FastAPI | Gradio |
|---|---|---|---|
| Main purpose | Data app / dashboard | API service | ML demo UI |
| Beginner UI work | Very little | You build or connect a UI separately | Very little |
| Other programs can call it? | Not its main purpose | Yes, that is the core use | Possible, but not the main focus |
| Best starting use | Small interactive app | Backend integration | Quick model demo |

## 8. Create `requirements.txt`

A deployment host needs to know which Python packages to install. In your project folder, create a file named `requirements.txt`:

```text
scikit-learn
joblib
numpy
streamlit
fastapi
uvicorn[standard]
gradio
```

**Why do we need this?** Your computer may already have packages installed, but a new server usually starts with a clean environment. A requirements file describes the dependencies.

For a more reproducible project, pin versions after you have tested them, for example by recording the environment's tested package versions. Avoid blindly pinning versions from a different environment.

## 9. Local testing checklist

Before sharing or hosting the app, test it.

- [ ] The saved model file exists.
- [ ] The model loads without an exception.
- [ ] Each interface starts successfully.
- [ ] Normal numeric inputs produce a prediction.
- [ ] Missing, malformed, zero, or negative inputs are handled or rejected appropriately.
- [ ] The input order and preprocessing match training.
- [ ] The prediction is sensible for known examples.
- [ ] The app does not expose secrets or private training data.

### Test the saved artifact without any UI

This is a quick way to find out whether a problem is in the model or in the app.

In [7]:
import joblib
import numpy as np

artifact = joblib.load("model.joblib")
model = artifact["model"]

sample = np.array([[5.1, 3.5, 1.4, 0.2]])
print("Prediction index:", int(model.predict(sample)[0]))

Prediction index: 0


## 10. Hosting: what changes when you go online?

Running on `localhost` means the app is on your own computer. Other people generally cannot access it from the public internet.

To make it public, choose a hosting provider that supports your framework:
- Streamlit app hosting for a Streamlit project;
- a Python web-service host or container platform for FastAPI;
- a supported Python app host for Gradio.

The exact steps vary by provider, so do not assume every host has the same buttons or settings. In general:

1. Put the code and `requirements.txt` in a Git repository or upload them as the provider requires.
2. Ensure `model.joblib` is included in the deployment, or securely downloaded from model storage during startup.
3. Set the correct start command for the framework.
4. Deploy and inspect build logs.
5. Test the public URL with normal and invalid inputs.
6. Set access controls if the service is not intended for everyone.

Example start commands:

```bash
# Streamlit
streamlit run app_streamlit.py --server.address 0.0.0.0 --server.port $PORT

# FastAPI (the hosting platform may provide its own PORT variable)
uvicorn app_fastapi:app --host 0.0.0.0 --port $PORT

# Gradio
python app_gradio.py
```

**Important:** `$PORT` is shell syntax commonly used on Linux-based hosts. It may not work as written on Windows or on every hosting service. Follow the selected provider's instructions. For Gradio, configure the port and host according to the platform's requirements.

### Public deployment safety

- Never commit API keys, passwords, private data, or credentials to Git.
- Do not load a model file from an unknown source.
- Consider authentication, rate limits, input limits, monitoring, and cost controls for a public API.
- A public URL means strangers may be able to send requests to your service.

## 11. Common errors and what they usually mean

| Error or symptom | Possible reason | First thing to check |
|---|---|---|
| `FileNotFoundError: model.joblib` | Model file isn't in the expected folder | Keep it beside the app file; check the path |
| `ModuleNotFoundError` | A package isn't installed in the active environment | Activate the environment and install requirements |
| Model says it expected 4 features | Wrong number of measurements | Send exactly four values |
| Strange predictions | Feature order or preprocessing changed | Match training feature order and reuse the pipeline |
| Port already in use | Another process uses that port | Stop the other process or choose another port |
| Works locally but fails when hosted | Missing files, wrong start command, package mismatch, or host-specific port | Read the build and runtime logs |
| API returns validation error | Request JSON is missing a field or contains invalid values | Check `/docs` and the expected schema |

### One important distinction

A **model error** means the model itself or its inputs are the problem. A **deployment error** means the app cannot start, load files, accept requests, or return responses. Test the model directly first, then test the interface.

## 12. Beginner exercises

Try these one at a time.

### Exercise 1 — Understand the artifact
1. Print `artifact_loaded.keys()`.
2. Explain why we saved `target_names` as well as the model.
3. What could happen if you swap the sepal length and sepal width inputs?

### Exercise 2 — Improve the Streamlit app
1. Add a short explanation of what the four measurements mean.
2. Add a section that displays the prediction index as well as the species.
3. Try changing the default values.

### Exercise 3 — Improve the API
1. Open `/docs`.
2. Send a valid prediction request.
3. Remove one field and send it again. What validation message do you receive?
4. Visit `/health` and explain why a health endpoint can be useful.

### Exercise 4 — Choose the right tool
For each scenario, choose Streamlit, FastAPI, or Gradio and explain why:
- A data analyst wants an interactive dashboard.
- A mobile app needs predictions through JSON.
- A student wants to demonstrate a model in a few minutes.

### Optional challenge
Replace the Iris example with your own trained scikit-learn model. Save the entire preprocessing pipeline and any label mapping needed to interpret predictions. Update the input fields to match the model's real features.

## 13. Final summary

- **Deployment** means making a trained model usable outside the training notebook.
- Save the **trained model and its preprocessing** together when possible.
- **Streamlit** is a good first choice for a small interactive Python app.
- **FastAPI** is a strong choice when other programs need to call your model.
- **Gradio** is excellent for quick interactive ML demos.
- Keep the model file, app code, and dependencies organized.
- Test the model directly and then test the full app.
- Public deployment requires extra care with security, input validation, versions, and hosting settings.

**Recommended learning order:** train and save the model → test loading it → build the Streamlit app → try Gradio → learn FastAPI → deploy one app online.

You do not need to master all three tools at once. Make one work locally first, understand why it works, and then move to the next.